In [1]:
import tensorflow as tf
import numpy as np
import cv2 # OpenCV for image processing
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image # For loading and preprocessing images

# --- Configuration ---
# IMPORTANT: Replace with the actual path to your trained model
MODEL_PATH = "C:/Users/OWNER/Downloads/SDS-CP028-smart-leaf/submissions/team-members/Samsudeen/my_model.keras" # Make sure you save your model after training!

# IMPORTANT: Replace with the path to the image you want to visualize
IMAGE_PATH = "C:/Users/OWNER/Downloads/SDS-CP028-smart-leaf/submissions/team-members/Samsudeen/TrainValTestDir/train/Rice___Leaf_Blast/aug_0_3452.jpg"

# IMPORTANT: Replace with the name of the last convolutional layer in your model.
# You can find this by printing model.summary() after your model is defined.
# For your current scratch CNN, it's likely 'conv2d_2' (if it's the third Conv2D layer)
LAST_CONV_LAYER_NAME = 'conv2d_2' # Example: 'conv2d_2' or 'conv2d_3' depending on your model

# Target image size used during model training
IMG_SIZE = (224, 224)
# --- End Configuration ---


def get_img_array(img_path, size):
    """Loads an image and preprocesses it for the model."""
    img = image.load_img(img_path, target_size=size)
    array = image.img_to_array(img)
    # Expand dimensions to create a batch of 1 image
    array = np.expand_dims(array, axis=0)
    return array / 255.0 # Rescale to 0-1, matching your ImageDataGenerator

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """
    Generates a Grad-CAM heatmap for a given image and model.

    Args:
        img_array (np.array): Preprocessed image array (batch of 1).
        model (tf.keras.Model): The trained Keras model.
        last_conv_layer_name (str): The name of the last convolutional layer.
        pred_index (int, optional): The index of the class for which to generate
                                    the heatmap. If None, the predicted class is used.

    Returns:
        np.array: The generated heatmap.
    """
    # Ensure the model's graph is built by running it on the input array.
    # This is crucial for accessing model.input and layer outputs when
    # creating the grad_model, especially for loaded Sequential models.
    _ = model(img_array)

    # Create a model that maps the input image to the activations of the last conv layer
    # and the final class predictions.
    # Use model.input (singular) as Sequential models typically have one input.
    grad_model = tf.keras.models.Model(
        inputs=model.input,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )

    # Use tf.GradientTape to compute gradients
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            # Get the index of the predicted class
            pred_index = tf.argmax(preds[0])
        # Get the loss for the predicted class
        class_channel = preds[:, pred_index]

    # Compute the gradient of the top predicted class with respect to the output
    # feature map of the last convolutional layer
    grads = tape.gradient(class_channel, last_conv_layer_output)

    # This is a vector where each entry is the mean intensity of the gradient
    # over a feature map channel
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Multiply each channel in the feature map array by "how important this channel is"
    # with regard to the predicted class
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # Normalize the heatmap between 0 and 1
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def display_gradcam(img_path, heatmap, alpha=0.4):
    """
    Overlays the heatmap on the original image and displays it.

    Args:
        img_path (str): Path to the original image.
        heatmap (np.array): The generated heatmap.
        alpha (float): Transparency factor for the heatmap overlay.
    """
    # Load the original image
    img = cv2.imread(img_path)
    img = cv2.resize(img, IMG_SIZE)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Convert to RGB for matplotlib

    # Resize heatmap to the original image size
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))

    # Convert heatmap to RGB
    heatmap = np.uint8(255 * heatmap)
    # Apply a colormap to the heatmap
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    # Superimpose the heatmap on the original image
    superimposed_img = heatmap * alpha + img
    superimposed_img = np.clip(superimposed_img, 0, 255).astype(np.uint8)

    # Display the image
    plt.figure(figsize=(8, 8))
    plt.imshow(superimposed_img)
    plt.axis('off')
    plt.title("Grad-CAM Heatmap")
    plt.show()

# --- Main execution ---
if __name__ == "__main__":
    try:
        # Load the trained model
        model = tf.keras.models.load_model(MODEL_PATH)
        print(f"Model loaded successfully from {MODEL_PATH}")

        # Explicitly build the model after loading to define input/output shapes
        # The input shape needs to match what the model expects, including batch dimension (None)
        model.build(input_shape=(None, *IMG_SIZE, 3))
        print("Model built successfully.")

        # The dummy prediction here is less critical now that model(img_array) is in make_gradcam_heatmap,
        # but it doesn't hurt to keep it for general robustness if needed elsewhere.
        # _ = model.predict(np.zeros((1, *IMG_SIZE, 3)))
        # print("Model graph confirmed to be built via dummy prediction.")


        # You can print the model summary to find the last convolutional layer name
        # model.summary()

        # Load and preprocess the image
        img_array = get_img_array(IMAGE_PATH, size=IMG_SIZE)

        # Generate the heatmap
        heatmap = make_gradcam_heatmap(img_array, model, LAST_CONV_LAYER_NAME)

        # Display the Grad-CAM result
        display_gradcam(IMAGE_PATH, heatmap)

        # Make a prediction to see what the model predicted for this image
        predictions = model.predict(img_array)
        predicted_class_index = np.argmax(predictions[0])
        # You'll need to get the class names from your generator or a saved mapping
        # For example, if you saved class_indices:
        # class_names = ['class_0', 'class_1', ...]
        # print(f"Model predicted: {class_names[predicted_class_index]} with confidence {predictions[0][predicted_class_index]:.2f}")

    except FileNotFoundError:
        print(f"Error: Model file not found at {MODEL_PATH}. Please train and save your model first.")
    except Exception as e:
        print(f"An error occurred: {e}")

Model loaded successfully from C:/Users/OWNER/Downloads/SDS-CP028-smart-leaf/submissions/team-members/Samsudeen/my_model.keras
Model built successfully.
An error occurred: The layer sequential has never been called and thus has no defined input.


C:\Users\OWNER\anaconda3\Lib\site-packages\keras\src\saving\saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 12 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
